In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
import random
import math

In [4]:
text = """
    the king loves the queen
    the king likes the queen
    the queen loves the king
    the queen likes the king
    the king rules the kingdom
    the queen rules the kingdom
    the king is strong
    the queen is beautiful
    """

In [6]:
tokens = text.lower().split()
len(tokens)

38

In [8]:
word_counts = Counter(tokens)
word_counts

Counter({'the': 14,
         'king': 6,
         'queen': 6,
         'loves': 2,
         'likes': 2,
         'rules': 2,
         'kingdom': 2,
         'is': 2,
         'strong': 1,
         'beautiful': 1})

In [12]:
vocab = sorted(word_counts.keys())
vocab

['beautiful',
 'is',
 'king',
 'kingdom',
 'likes',
 'loves',
 'queen',
 'rules',
 'strong',
 'the']

In [16]:
word_to_idx = {word: i for i, word in enumerate(vocab)}
word_to_idx

{'beautiful': 0,
 'is': 1,
 'king': 2,
 'kingdom': 3,
 'likes': 4,
 'loves': 5,
 'queen': 6,
 'rules': 7,
 'strong': 8,
 'the': 9}

In [30]:
word_to_idx['rules']

7

In [25]:
idx_to_word = {i: word for word, i in word_to_idx.items()}
idx_to_word

{0: 'beautiful',
 1: 'is',
 2: 'king',
 3: 'kingdom',
 4: 'likes',
 5: 'loves',
 6: 'queen',
 7: 'rules',
 8: 'strong',
 9: 'the'}

In [31]:
idx_to_word[7]

'rules'

In [28]:
vocab_size = len(vocab)
vocab_size


10

In [33]:
corpus = [word_to_idx[word] for word in tokens]
corpus[:5]

[9, 2, 5, 9, 6]

In [34]:
len(corpus)

38

In [37]:
def generate_skipgram_pairs(corpus, window_size):

    pairs = []

    for center_position in range(len(corpus)):

        center_word = corpus[center_position]
        start = max(0, center_position - window_size)
        end = min(len(corpus), center_position + window_size + 1)

        for context_position in range(start, end):

            if context_position == center_position:
                continue

            context_word = corpus[context_position]
            pairs.append((center_word, context_word))

    return pairs



In [40]:
window_size = 2

pairs = generate_skipgram_pairs(corpus, window_size)
pairs[:5]

[(9, 2), (9, 5), (2, 9), (2, 5), (2, 9)]

In [41]:
len(pairs)

146

In [43]:
38*4 - (2*2 + 1*2)

146

In [44]:
for center, context in pairs[:5]:
    print(idx_to_word[center], '-->', idx_to_word[context])

the --> king
the --> loves
king --> the
king --> loves
king --> the


In [48]:
word_counts['the']

14

In [56]:
frequencies = torch.tensor([word_counts[idx_to_word[i]] for i in range(vocab_size)], dtype=torch.float)
frequencies

tensor([ 1.,  2.,  6.,  2.,  2.,  2.,  6.,  2.,  1., 14.])

In [57]:
weights = frequencies ** 0.75
weights

tensor([1.0000, 1.6818, 3.8337, 1.6818, 1.6818, 1.6818, 3.8337, 1.6818, 1.0000,
        7.2376])

In [58]:
probabilities = weights / weights.sum()
probabilities

tensor([0.0395, 0.0664, 0.1514, 0.0664, 0.0664, 0.0664, 0.1514, 0.0664, 0.0395,
        0.2859])

In [ ]:
class SkipGramNegativeSampling(nn.Module):

    def __init__(self, vocab_size, embedding_dim):

        super().__init__()
        self.in_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.out_embedding = nn.Embedding(vocab_size, embedding_dim)